In [6]:
import pandas as pd
import time

In [1]:
#pip install google.generativeai

In [ ]:
import google.generativeai as genai

genai.configure(api_key = "YOUR_API_KEY_HERE")
aimodel   = "gemini-3.5-flash-lite"

In [8]:
#Loading the data

df = pd.read_csv("ai_teacher_20_questions.csv")
print(f"Total questions: {len(df)}")

Total questions: 20


In [9]:
# AI TEACHER PROMPT

instructions = """
You are an expert AI Teacher.

Explain the concept using simple layman terms.
Keep the answer precise and easy to understand.
Give the answer in approximately 3 lines.
"""


In [25]:
#Storage

answers = []
latencies = []
input_tokens = []
output_tokens = []
total_tokens = []
input_costs = []
output_costs = []
total_costs = []

In [23]:
# Price per 1 million tokens
INPUT_PRICE = 0.20
OUTPUT_PRICE = 1.20

In [26]:
# Generate answers

for index, row in df.iterrows():

  question = row['User Query']
  full_prompt = f"{instructions}\n\nConcept: {question}"

  try:
    # START TIMER
    start_time = time.perf_counter()

    model    = genai.GenerativeModel(aimodel)
    response = model.generate_content(full_prompt)

    # END TIMER
    end_time = time.perf_counter()

    # LATENCY
    latency = end_time - start_time

    # ANSWER
    answer = response.text

    # TOKENS
    input_token_count = response.usage_metadata.prompt_token_count
    output_token_count = response.usage_metadata.candidates_token_count
    total_token_count = response.usage_metadata.total_token_count

    # COST
    input_cost = (input_token_count / 1_000_000) * INPUT_PRICE
    output_cost = (output_token_count / 1_000_000) * OUTPUT_PRICE
    total_cost = input_cost + output_cost

    # STORE VALUES
    answers.append(answer)
    latencies.append(latency)
    input_tokens.append(input_token_count)
    output_tokens.append(output_token_count)
    total_tokens.append(total_token_count)
    input_costs.append(input_cost)
    output_costs.append(output_cost)
    total_costs.append(total_cost)

  except Exception as e:
    print(f"Error processing row {index}: {e}")

    answers.append("ERROR")
    latencies.append(0)
    input_tokens.append(0)
    output_tokens.append(0)
    total_tokens.append(0)
    input_costs.append(0)
    output_costs.append(0)
    total_costs.append(0)


Error processing row 11: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.
Please retry in 40.381046907s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://aistudio.google.com/docs/rate-limits"
}
, violations {
  description: "Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimens

In [27]:
 #Adding Results to DataFrame

df["GeneratedAnswer"] = answers
df["Latency"]         = latencies
df["InputTokens"]     = input_tokens
df["OutputTokens"]    = output_tokens
df["TotalTokens"]     = total_tokens
df["InputCost"]       = input_costs
df["OutputCost"]      = output_costs
df["TotalCost"]       = total_costs

In [28]:
#Display

df[
    [
        "User Query",
        "GeneratedAnswer",
        "Latency",
        "InputTokens",
        "OutputTokens",
        "TotalTokens",
        "InputCost",
        "OutputCost",
        "TotalCost"
    ]
]

,User Query,GeneratedAnswer,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,"AI, or Artificial Intelligence, is like teachi...",0.692357,45,60,105,0.000009,0.000072,0.000081
1,How does AI differ from traditional programming?,Traditional programming is like following a st...,0.647259,49,79,128,0.000010,0.000095,0.000105
2,What is Machine Learning?,Machine Learning is like teaching a computer t...,0.782341,46,64,110,0.000009,0.000077,0.000086
3,What is Deep Learning?,Deep learning is a smart computer technique in...,0.729968,46,69,115,0.000009,0.000083,0.000092
4,"What is the difference between AI, ML, and Dee...",Think of **AI (Artificial Intelligence)** as t...,0.667711,54,83,137,0.000011,0.000100,0.000110
5,What is training data?,Training data is essentially a collection of e...,0.839836,46,72,118,0.000009,0.000086,0.000096
6,What is a Machine Learning model?,A Machine Learning model is like a digital rec...,0.797782,48,66,114,0.000010,0.000079,0.000089
7,What is supervised learning?,Supervised learning is like teaching a compute...,0.562645,46,64,110,0.000009,0.000077,0.000086
8,What is unsupervised learning?,Imagine teaching a child by giving them a big ...,1.590962,46,68,114,0.000009,0.000082,0.000091
9,What is reinforcement learning?,Reinforcement learning is a type of AI trainin...,0.656155,46,60,106,0.000009,0.000072,0.000081


In [29]:
# Saving Results

df.to_csv("ai_teacher_evaluation_Operational_metrics.csv", index = False)

In [30]:
#Loading the results again and performing averge & sum
df2 = pd.read_csv("ai_teacher_evaluation_Operational_metrics.csv")

In [33]:
metric_columns = ["Latency", "InputTokens", "OutputTokens", "TotalTokens", "InputCost", "OutputCost", "TotalCost" ]

# AVERAGE METRICS

average_latency = df["Latency"].mean()
average_input_tokens = df["InputTokens"].mean()
average_output_tokens = df["OutputTokens"].mean()
average_total_tokens = df["TotalTokens"].mean()
average_input_cost = df["InputCost"].mean()
average_output_cost = df["OutputCost"].mean()
average_total_cost = df["TotalCost"].mean()

# TOTAL METRICS

total_latency = df["Latency"].sum()
total_input_tokens = df["InputTokens"].sum()
total_output_tokens = df["OutputTokens"].sum()
total_tokens = df["TotalTokens"].sum()
total_input_cost = df["InputCost"].sum()
total_output_cost = df["OutputCost"].sum()
total_cost = df["TotalCost"].sum()

In [32]:
for column in metric_columns:
    df2[column] = pd.to_numeric(df2[column], errors="coerce")

In [34]:
# Creating Operational metrics dataframe

operational_metrics_df = pd.DataFrame({
    "Metric" : ["Latency (seconds)", "Input Tokens", "Output Tokens", "Total Tokens", "Input Cost (USD)", "Output Cost (USD)", "Total Cost (USD)"],
    "Average": [average_latency, average_input_tokens, average_output_tokens, average_total_tokens, average_input_cost, average_output_cost, average_total_cost],
    "Total"  : [total_latency, total_input_tokens, total_output_tokens, total_tokens, total_input_cost, total_output_cost, total_cost]
})

In [35]:
# Round up the values

operational_metrics_df["Average"] = (operational_metrics_df["Average"].round(6))
operational_metrics_df["Total"] = (operational_metrics_df["Total"].round(6))

# DISPLAY DATAFRAME
operational_metrics_df

,Metric,Average,Total
0,Latency (seconds),0.706571,14.131424
1,Input Tokens,42.150000,843.000000
2,Output Tokens,63.000000,1260.000000
3,Total Tokens,105.150000,2103.000000
4,Input Cost (USD),0.000008,0.000169
5,Output Cost (USD),0.000076,0.001512
6,Total Cost (USD),0.000084,0.001681


In [36]:
operational_metrics_df.to_csv("ai_teacher_evaluation_Operational_metrics_average.csv", index=False, encoding="utf-8-sig")